# Improved Car Price Prediction: PEUGEOT_208 Analysis
## Comprehensive ML Study Following Expert Recommendations

This notebook implements key improvements based on ML expert feedback, focusing specifically on **PEUGEOT_208** to match the original study methodology:
- **Outlier detection and removal** using multiple statistical methods
- **Enhanced feature engineering** with proper scaling and new features
- **Hyperparameter tuning** using GridSearchCV
- **Cross-validation** for robust model evaluation
- **Advanced models** including XGBoost
- **Comprehensive evaluation** with multiple metrics and residual analysis
- **Statistical significance testing** for model comparisons

**Dataset**: Car listings from leboncoin.fr stored in DuckDB
**Target Model**: PEUGEOT_208 (consistent with original study)
**Target**: Predict car prices using age, mileage, and engineered features
**Approach**: Rigorous ML methodology with proper validation

In [ ]:
# Import Required Libraries
import sys
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append(os.path.abspath('../src'))

# Data handling and analysis
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import zscore, probplot
import sqlite3

# Database connection
from data.database import get_car_items

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error, 
                           mean_absolute_percentage_error)
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.base import BaseEstimator, RegressorMixin
from scipy.optimize import curve_fit, minimize

# Advanced ML models
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not available. Install with: pip install xgboost")

# Statistical tests
from scipy.stats import ttest_rel, wilcoxon

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("All libraries imported successfully!")
print(f"XGBoost available: {XGB_AVAILABLE}")

In [ ]:
# Utility Functions to Eliminate Duplication
from typing import Tuple, Dict, List, Any
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

# Data Processing Utilities
def calculate_data_quality_score(data: pd.DataFrame, columns: List[str]) -> float:
    """Calculate a data quality score based on skewness and outliers"""
    scores = []
    for col in columns:
        # Skewness penalty (lower skew = better quality)
        skewness_penalty = abs(data[col].skew()) / 10
        
        # Outlier penalty using IQR
        Q1, Q3 = data[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        outliers = data[(data[col] < Q1 - 1.5*IQR) | (data[col] > Q3 + 1.5*IQR)]
        outlier_penalty = len(outliers) / len(data)
        
        # Combined score (higher is better)
        score = max(0, 1 - skewness_penalty - outlier_penalty)
        scores.append(score)
    
    return np.mean(scores)

def create_statistical_summary(data: pd.DataFrame, columns: List[str], title: str) -> pd.DataFrame:
    """Create a standardized statistical summary table"""
    summary_stats = []
    for column in columns:
        stats = data[column].describe()
        summary_stats.append([
            column.replace('_', ' ').title(),
            f"{stats['mean']:.1f}",
            f"{stats['std']:.1f}", 
            f"{stats['min']:.0f}",
            f"{stats['max']:.0f}",
            f"{stats['50%']:.1f}"  # median
        ])
    
    return pd.DataFrame(summary_stats, 
                       columns=['Feature', 'Mean', 'Std', 'Min', 'Max', 'Median'])

# Outlier Detection Utilities
def detect_outliers_iqr(data: pd.DataFrame, column: str, factor: float = 1.5) -> Tuple[pd.DataFrame, float, float]:
    """Detect outliers using Interquartile Range (IQR) method"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

def detect_outliers_zscore(data: pd.DataFrame, column: str, threshold: float = 3) -> Tuple[pd.DataFrame, np.ndarray]:
    """Detect outliers using Z-score method"""
    z_scores = np.abs(zscore(data[column]))
    outliers = data[z_scores > threshold]
    return outliers, z_scores

def detect_outliers_modified_zscore(data: pd.DataFrame, column: str, threshold: float = 3.5) -> Tuple[pd.DataFrame, np.ndarray]:
    """Detect outliers using Modified Z-score method (more robust)"""
    median = data[column].median()
    mad = np.median(np.abs(data[column] - median))  # Median Absolute Deviation
    modified_z_scores = 0.6745 * (data[column] - median) / mad
    outliers = data[np.abs(modified_z_scores) > threshold]
    return outliers, modified_z_scores

def analyze_outliers_for_columns(data: pd.DataFrame, columns: List[str]) -> Dict[str, Dict[str, Any]]:
    """Apply all outlier detection methods to multiple columns"""
    outlier_results = {}
    
    for column in columns:
        # IQR method
        iqr_outliers, lower_iqr, upper_iqr = detect_outliers_iqr(data, column)
        
        # Z-score method
        zscore_outliers, z_scores = detect_outliers_zscore(data, column)
        
        # Modified Z-score method
        mod_zscore_outliers, mod_z_scores = detect_outliers_modified_zscore(data, column)
        
        outlier_results[column] = {
            'iqr': {
                'outliers': iqr_outliers,
                'count': len(iqr_outliers),
                'bounds': (lower_iqr, upper_iqr)
            },
            'zscore': {
                'outliers': zscore_outliers,
                'count': len(zscore_outliers),
                'scores': z_scores
            },
            'mod_zscore': {
                'outliers': mod_zscore_outliers,
                'count': len(mod_zscore_outliers),
                'scores': mod_z_scores
            }
        }
    
    return outlier_results

# Model Training and Evaluation Utilities
def mean_absolute_percentage_error_custom(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Custom MAPE with handling for zero values"""
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def calculate_model_metrics(y_true: pd.Series, y_pred: np.ndarray) -> Dict[str, float]:
    """Calculate all model evaluation metrics consistently"""
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error_custom(y_true, y_pred)
    }

def perform_cross_validation_comprehensive(model, X_train: pd.DataFrame, y_train: pd.Series, cv=None) -> Dict[str, Dict[str, Any]]:
    """Perform comprehensive cross-validation with multiple metrics"""
    from sklearn.model_selection import TimeSeriesSplit
    
    if cv is None:
        cv = TimeSeriesSplit(n_splits=5)
    
    # Cross-validation scores
    cv_scores = {}
    
    # R² scores
    r2_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1)
    cv_scores['r2_score'] = {'scores': r2_scores, 'mean': r2_scores.mean(), 'std': r2_scores.std()}
    
    # MSE scores (negative because sklearn returns negative MSE)
    mse_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    cv_scores['neg_mse'] = {'scores': mse_scores, 'mean': mse_scores.mean(), 'std': mse_scores.std()}
    
    # MAE scores
    mae_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    cv_scores['neg_mae'] = {'scores': mae_scores, 'mean': mae_scores.mean(), 'std': mae_scores.std()}
    
    # MAPE scores (custom implementation)
    mape_scores = []
    for train_idx, val_idx in cv.split(X_train):
        X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Train on fold
        fold_model = clone(model)
        fold_model.fit(X_train_fold, y_train_fold)
        
        # Predict on validation
        y_pred_fold = fold_model.predict(X_val_fold)
        mape_fold = mean_absolute_percentage_error_custom(y_val_fold.values, y_pred_fold)
        mape_scores.append(-mape_fold)  # Negative to match sklearn convention
    
    mape_scores = np.array(mape_scores)
    cv_scores['neg_mape'] = {'scores': mape_scores, 'mean': mape_scores.mean(), 'std': mape_scores.std()}
    
    return cv_scores

# Visualization Utilities
def create_distribution_plots(data: pd.DataFrame, columns: List[str], title_prefix: str = "") -> None:
    """Create standardized distribution plots for multiple columns"""
    fig, axes = plt.subplots(2, len(columns), figsize=(6*len(columns), 12))
    if title_prefix:
        fig.suptitle(f'{title_prefix}: Distribution Analysis', fontsize=16, fontweight='bold')
    
    colors = ['lightcoral', 'lightgreen', 'salmon']
    
    for i, column in enumerate(columns):
        # Raw distribution
        axes[0, i].hist(data[column], bins=50, alpha=0.7, color=colors[i % len(colors)], edgecolor='black')
        axes[0, i].set_title(f'{column.replace("_", " ").title()} - Raw')
        axes[0, i].set_xlabel(column.replace("_", " ").title())
        axes[0, i].set_ylabel('Frequency')
        
        # Add mean and median lines
        mean_val = data[column].mean()
        median_val = data[column].median()
        axes[0, i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.1f}')
        axes[0, i].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.1f}')
        axes[0, i].legend()
        
        # Log-transformed distribution
        axes[1, i].hist(np.log1p(data[column]), bins=50, alpha=0.7, color=colors[i % len(colors)], edgecolor='black')
        axes[1, i].set_title(f'{column.replace("_", " ").title()} - Log-transformed')
        axes[1, i].set_xlabel(f'log({column.replace("_", " ").title()} + 1)')
        axes[1, i].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

def print_dataset_summary(data: pd.DataFrame, title: str, target_model: str = None) -> None:
    """Print standardized dataset summary information"""
    print(f"\n📊 {title}")
    print("="*60)
    if target_model:
        print(f"🎯 Target Model: {target_model}")
    print(f"📈 Records: {len(data):,}")
    print(f"🔧 Features: {data.shape[1]}")
    
    if 'price' in data.columns:
        print(f"💰 Price range: €{data['price'].min():,.0f} - €{data['price'].max():,.0f}")
    if 'age_months' in data.columns:
        print(f"📅 Age range: {data['age_months'].min():.0f} - {data['age_months'].max():.0f} months")
    if 'mileage_kkms' in data.columns:
        print(f"🚗 Mileage range: {data['mileage_kkms'].min():.0f} - {data['mileage_kkms'].max():.0f} kkms")

print("✅ Utility functions loaded successfully!")
print("🔧 These functions will eliminate code duplication throughout the notebook")

In [ ]:
# declare a simple model that will use a scikit curve_fit to find the best parameters
# price = α × (base_lin - a_lin×mileage - b_lin×age) + (1-α) × base_exp × exp(-a_exp×mileage) × exp(-b_exp×age)

from sklearn.base import BaseEstimator, RegressorMixin

class CurveFitModel(BaseEstimator, RegressorMixin):
    def __init__(self):
        """Initialize CurveFitModel with default parameters that will be optimized during fit"""
        self.base_lin = None
        self.a_lin = None
        self.b_lin = None
        self.base_exp = None
        self.a_exp = None
        self.b_exp = None
        self.is_fitted = False

    def get_params(self, deep=True):
        """Get parameters for this estimator (required for sklearn compatibility)"""
        return {}

    def set_params(self, **params):
        """Set parameters for this estimator (required for sklearn compatibility)"""
        for key, value in params.items():
            if hasattr(self, key):
                setattr(self, key, value)
        return self

    def _mixed_model_func(self, X, base_lin, a_lin, b_lin, base_exp, a_exp, b_exp):
        """Mixed exponential-linear model function for curve fitting"""
        mileage, age = X[0], X[1]
        linear_part = base_lin - a_lin * mileage - b_lin * age
        exp_part = base_exp * np.exp(-a_exp * mileage) * np.exp(-b_exp * age)
        return linear_part + exp_part

    def fit(self, X, y):
        """
        Fit the mixed model using curve fitting to optimize all parameters
        
        Args:
            X: Training features with 'mileage_kkms' and 'age_months' columns
            y: Training target (price)
        """
        from scipy.optimize import curve_fit
        
        # Extract mileage_kkms and age_months directly
        mileage_data = X['mileage_kkms'].values
        age_data = X['age_months'].values
        
        # Prepare data for curve fitting
        X_curve = np.array([mileage_data, age_data])
        y_values = y.values if hasattr(y, 'values') else y
        
        # Initial parameter guesses
        mean_price = np.mean(y_values)
        p0 = [
            mean_price/2,      # base_lin
            mean_price/150,  # mean_price is lost after 150kkms
            mean_price/120,  # mean_price is lost after 120 months
            mean_price/2,      # base_exp
            1.0/150,  # a_exp
            1.0/120,  # b_exp
        ]

         # Set parameter bounds
        bounds = (
            [1.0, 0.0, 0.0, 1.0, 0.0, 0.0],    # lower bounds
            [10*mean_price, mean_price, mean_price, 10*mean_price, 1.0, 1.0]       # upper bounds
        )
        
        # Perform curve fitting
        popt, _ = curve_fit(
            self._mixed_model_func,
            X_curve,
            y_values,
            p0=p0,
            bounds=bounds
        )
        
        # Store optimized parameters
        self.base_lin, self.a_lin, self.b_lin = popt[0], popt[1], popt[2]
        self.base_exp, self.a_exp, self.b_exp = popt[3], popt[4], popt[5]
        self.is_fitted = True
        
        return self

    def predict(self, X):
        """
        Predict using the fitted mixed model
        
        Args:
            X: Features with 'mileage_kkms' and 'age_months' columns
            
        Returns:
            Predicted prices
        """
        if not self.is_fitted:
            raise ValueError("Model must be fitted before making predictions")
        
        # Extract mileage_kkms and age_months directly
        mileage = X['mileage_kkms'].values
        age = X['age_months'].values
        
        # Calculate prediction using mixed model
        return self._mixed_model_func((mileage, age), self.base_lin, self.a_lin, self.b_lin, self.base_exp, self.a_exp, self.b_exp)

## 1. Load and Explore Dataset
Load data from DuckDB and perform comprehensive initial exploration to understand data quality and structure.

In [ ]:
# Load and process the data
df_raw = get_car_items()

# Data validation and cleaning
print(f"Initial data shape: {df_raw.shape}")

# Remove rows with missing values in critical columns
df_raw = df_raw.dropna(subset=['model', 'age_in_days', 'mileage', 'price'])

# Feature transformations
df_raw['mileage_kkms'] = df_raw['mileage'] / 1000.0
df_raw = df_raw.drop(columns=['mileage'])

df_raw['age_months'] = df_raw['age_in_days'] / (365.25 / 12)  # Convert days to months
df_raw = df_raw.drop(columns=['age_in_days'])

# Print comprehensive dataset summary using utility function
print_dataset_summary(df_raw, "DATASET OVERVIEW")

print(f"\n📊 Feature Information:")
print(df_raw.info())

# Check missing values
missing_values = df_raw.isnull().sum()
missing_pct = (missing_values / len(df_raw)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_pct
})
print(f"\n📊 Missing Values:")
print(missing_df[missing_df['Missing Count'] > 0])

# Model distribution overview
model_counts_all = df_raw['model'].value_counts()
print(f"\n📊 MODEL DISTRIBUTION:")
print(f"   • Total unique models: {len(model_counts_all)}")
print(f"   • Records per model range: {model_counts_all.min()} - {model_counts_all.max()}")
print(f"\n📊 Top 10 most common models:")
print(model_counts_all.head(10))

print("\n✅ Data loading complete! Proceed to model selection.")

## Model Selection - Focus on PEUGEOT_208
Select the target car model for analysis. This is done early to ensure all subsequent analysis focuses on a single model for consistency.

In [ ]:
# 🎯 CONFIGURABLE MODEL SELECTION
# =================================
# Change this variable to analyze a different car model
TARGET_MODEL = 'PEUGEOT_208'  # 👈 CHANGE THIS TO SELECT DIFFERENT MODEL

print("🚗 MODEL SELECTION")
print("="*50)
print(f"Target model: {TARGET_MODEL}")

# Show available models for reference
print("\nTop 15 most common models in dataset:")
model_counts_all = df_raw['model'].value_counts()
print(model_counts_all.head(15).to_string())

# Check if target model exists and filter dataset
if TARGET_MODEL in model_counts_all.index:
    df_filtered = df_raw[df_raw['model'] == TARGET_MODEL].copy()
    records_available = len(df_filtered)
    print(f"\n✅ Found {records_available:,} records for {TARGET_MODEL}")
    
    if records_available < 100:
        print(f"⚠️  Warning: Only {records_available} records available for {TARGET_MODEL}")
        print("   Consider selecting a model with more data for better analysis")
    
    # Update the main dataframe to work with selected model only
    df_raw = df_filtered.copy()
    
    print(f"\n📊 Dataset after filtering to {TARGET_MODEL}:")
    print(f"   • Records: {len(df_raw):,}")
    print(f"   • Features: {df_raw.shape[1]}")
    print(f"   • Price range: €{df_raw['price'].min():,.0f} - €{df_raw['price'].max():,.0f}")
    print(f"   • Age range: {df_raw['age_months'].min()} - {df_raw['age_months'].max()} months")
    print(f"   • Mileage range: {df_raw['mileage_kkms'].min():,.0f} - {df_raw['mileage_kkms'].max():,.0f} kkms")
    
else:
    print(f"\n❌ Error: Model '{TARGET_MODEL}' not found in dataset!")
    print("\nAvailable models:")
    print(model_counts_all.head(20).to_string())
    print(f"\n💡 Tip: Copy one of the model names above and update TARGET_MODEL variable")
    
    # Stop execution to prevent errors
    raise ValueError(f"Model '{TARGET_MODEL}' not found. Please update TARGET_MODEL variable.")

print(f"\n✅ Analysis will focus exclusively on {TARGET_MODEL}")
print("="*50)

### 📝 Note: Easy Model Configuration
The `TARGET_MODEL` variable at the top of this section makes it simple to analyze different car models:
- **Current focus**: PEUGEOT_208 (matching original study)
- **To change**: Simply update `TARGET_MODEL = 'YOUR_MODEL_NAME'` 
- **Benefits**: All subsequent analysis automatically adapts to the selected model
- **Validation**: The code checks model availability and displays available options

This approach ensures consistent, single-model analysis while maintaining flexibility for different research questions.

In [ ]:
# Comprehensive Data Exploration using utility functions
numerical_cols = ['price', 'age_months', 'mileage_kkms']

# Create standardized distribution plots
create_distribution_plots(df_raw, numerical_cols, "Raw Data")

# Correlation analysis
correlation_matrix = df_raw[numerical_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Matrix')
plt.show()

# Print correlation insights
print("📊 CORRELATION ANALYSIS")
print("="*40)
print(f"Price vs Age: {correlation_matrix.loc['price', 'age_months']:.3f}")
print(f"Price vs Mileage: {correlation_matrix.loc['price', 'mileage_kkms']:.3f}")
print(f"Age vs Mileage: {correlation_matrix.loc['age_months', 'mileage_kkms']:.3f}")

# Statistical summary using utility function
stats_table = create_statistical_summary(df_raw, numerical_cols, "Basic Statistics")
print(f"\n📊 STATISTICAL SUMMARY:")
print(stats_table.to_string(index=False))

## 2. Detect Outliers Using Statistical Methods
Implement multiple outlier detection techniques to identify potentially problematic data points that could negatively impact model performance.

In [ ]:
# Apply comprehensive outlier detection using utility functions
numerical_columns = ['price', 'age_months', 'mileage_kkms']
outlier_results = analyze_outliers_for_columns(df_raw, numerical_columns)

print("OUTLIER DETECTION RESULTS")
print("="*70)

# Print detailed results for each column
for column in numerical_columns:
    print(f"\n📊 ANALYZING: {column.upper()}")
    print("-" * 50)
    
    results = outlier_results[column]
    bounds = results['iqr']['bounds']
    
    print(f"IQR Method (bounds: {bounds[0]:.1f} to {bounds[1]:.1f}):")
    print(f"  • Outliers detected: {results['iqr']['count']:,} ({results['iqr']['count']/len(df_raw)*100:.2f}%)")
    
    print(f"Z-score Method (threshold: 3.0):")
    print(f"  • Outliers detected: {results['zscore']['count']:,} ({results['zscore']['count']/len(df_raw)*100:.2f}%)")
    
    print(f"Modified Z-score Method (threshold: 3.5):")
    print(f"  • Outliers detected: {results['mod_zscore']['count']:,} ({results['mod_zscore']['count']/len(df_raw)*100:.2f}%)")

# Create summary table
print(f"\n📊 OUTLIER DETECTION SUMMARY")
print("="*70)
summary_table = [
    ["Feature", "IQR Count", "IQR %", "Z-score Count", "Z-score %", "Mod Z-score Count", "Mod Z-score %"],
    ["-------", "----------", "------", "-------------", "---------", "----------------", "-------------"]
]

for column in numerical_columns:
    results = outlier_results[column]
    summary_table.append([
        column,
        f"{results['iqr']['count']:,}",
        f"{results['iqr']['count']/len(df_raw)*100:.2f}%",
        f"{results['zscore']['count']:,}",
        f"{results['zscore']['count']/len(df_raw)*100:.2f}%",
        f"{results['mod_zscore']['count']:,}",
        f"{results['mod_zscore']['count']/len(df_raw)*100:.2f}%"
    ])

for row in summary_table:
    print(f"{row[0]:<12} {row[1]:<10} {row[2]:<6} {row[3]:<13} {row[4]:<9} {row[5]:<16} {row[6]:<13}")

print(f"\n✅ Outlier detection complete using standardized utility functions")

## 3. Visualize Outliers with Box Plots and Scatter Plots
Create comprehensive visualizations to understand outlier patterns and their distribution across features.

In [ ]:
# Comprehensive Outlier Visualization using efficient plotting
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1])
fig.suptitle('Outlier Detection Analysis', fontsize=16, fontweight='bold')

columns_viz = ['price', 'age_months', 'mileage_kkms']

# Box plots for each feature
for i, column in enumerate(columns_viz):
    ax = fig.add_subplot(gs[0, i])
    
    # Create box plot with consistent styling
    bp = ax.boxplot([df_raw[column]], labels=[column.replace('_', ' ').title()], patch_artist=True)
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][0].set_alpha(0.7)
    
    ax.set_title(f'{column.replace("_", " ").title()} - Box Plot')
    ax.grid(True, alpha=0.3)
    
    # Add outlier statistics using pre-calculated results
    outlier_count = outlier_results[column]['iqr']['count']
    ax.text(0.02, 0.98, f'Outliers: {outlier_count} ({outlier_count/len(df_raw)*100:.1f}%)', 
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Scatter plots showing relationships with outliers
scatter_combinations = [('age_months', 'price'), ('mileage_kkms', 'price'), ('age_months', 'mileage_kkms')]

for i, (x_col, y_col) in enumerate(scatter_combinations):
    ax = fig.add_subplot(gs[1, i])
    
    # Plot normal points
    ax.scatter(df_raw[x_col], df_raw[y_col], alpha=0.5, s=20, color='blue', label='Normal')
    
    # Highlight outliers using pre-calculated results
    x_outliers = outlier_results[x_col]['iqr']['outliers'] if x_col in outlier_results else pd.DataFrame()
    y_outliers = outlier_results[y_col]['iqr']['outliers'] if y_col in outlier_results else pd.DataFrame()
    
    # Plot outliers with different colors
    if not x_outliers.empty:
        ax.scatter(x_outliers[x_col], x_outliers[y_col], alpha=0.7, s=30, color='red', label=f'{x_col} outliers')
    
    if not y_outliers.empty:
        ax.scatter(y_outliers[x_col], y_outliers[y_col], alpha=0.7, s=30, color='orange', label=f'{y_col} outliers')
    
    ax.set_xlabel(x_col.replace('_', ' ').title())
    ax.set_ylabel(y_col.replace('_', ' ').title())
    ax.set_title(f'{x_col.replace("_", " ").title()} vs {y_col.replace("_", " ").title()}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show extreme cases for manual inspection using consistent format
print("\n🔍 EXTREME OUTLIERS FOR MANUAL INSPECTION")
print("="*60)

inspection_columns = ['model', 'price', 'age_months', 'mileage_kkms']

print("\n Most Expensive Cars:")
expensive_cars = df_raw.nlargest(5, 'price')[inspection_columns]
print(expensive_cars.to_string(index=False))

print("\n📊 Lowest Priced Cars:")
cheap_cars = df_raw.nsmallest(5, 'price')[inspection_columns]
print(cheap_cars.to_string(index=False))

print("\n📊 Highest Mileage Cars:")
high_mileage = df_raw.nlargest(5, 'mileage_kkms')[inspection_columns]
print(high_mileage.to_string(index=False))

print("\n✅ Outlier visualization complete")

## 4. Remove Outliers Using IQR Method
Apply the IQR method to remove outliers systematically while preserving data integrity.

In [ ]:
# Remove outliers using IQR method with utility functions
def remove_outliers_iqr_comprehensive(data: pd.DataFrame, columns: List[str], factor: float = 1.5) -> Tuple[pd.DataFrame, Dict]:
    """Remove outliers using IQR method for multiple columns with detailed stats"""
    df_clean = data.copy()
    removal_stats = {}
    
    print("REMOVING OUTLIERS USING IQR METHOD")
    print("="*60)
    
    for column in columns:
        original_count = len(df_clean)
        
        # Use utility function for outlier detection
        outliers, lower_bound, upper_bound = detect_outliers_iqr(df_clean, column, factor)
        
        # Remove outliers
        df_clean = df_clean[(df_clean[column] >= lower_bound) & (df_clean[column] <= upper_bound)]
        
        removed_count = original_count - len(df_clean)
        removal_percentage = (removed_count / original_count) * 100
        
        removal_stats[column] = {
            'original_count': original_count,
            'removed_count': removed_count,
            'remaining_count': len(df_clean),
            'removal_percentage': removal_percentage,
            'bounds': (lower_bound, upper_bound)
        }
        
        print(f"📊 {column.upper()}:")
        print(f"   • Bounds: [{lower_bound:.0f}, {upper_bound:.0f}]")
        print(f"   • Removed: {removed_count:,} records ({removal_percentage:.2f}%)")
        print(f"   • Remaining: {len(df_clean):,} records")
        print()
    
    return df_clean, removal_stats

# Apply IQR outlier removal
numerical_columns = ['price', 'age_months', 'mileage_kkms']
df_clean_iqr, iqr_removal_stats = remove_outliers_iqr_comprehensive(df_raw, numerical_columns, factor=1.5)

print(f"📈 OVERALL IMPACT:")
print(f"   • Original dataset: {len(df_raw):,} records")
print(f"   • Clean dataset: {len(df_clean_iqr):,} records")
total_removed = len(df_raw) - len(df_clean_iqr)
print(f"   • Total removed: {total_removed:,} records ({(total_removed/len(df_raw))*100:.2f}%)")
print(f"   • Data retention: {(len(df_clean_iqr)/len(df_raw))*100:.2f}%")

# Apply domain-specific filters
print("\n" + "="*60)
print("APPLYING DOMAIN-SPECIFIC FILTERS")
print("="*60)

original_clean_count = len(df_clean_iqr)

# Domain filters with tracking
domain_filters = [
    ("cheap cars with low mileage", lambda df: ~((df['price'] < 2000) & (df['mileage_kkms'] < 10000))),
    ("cars older than 25 years", lambda df: df['age_months'] <= 25 * 12),
    ("cars with mileage > 500,000km", lambda df: df['mileage_kkms'] <= 500000)
]

for filter_name, filter_func in domain_filters:
    before_count = len(df_clean_iqr)
    df_clean_iqr = df_clean_iqr[filter_func(df_clean_iqr)]
    removed = before_count - len(df_clean_iqr)
    print(f"🚗 Removed {filter_name}: {removed:,}")

# Store the final clean dataset
df_clean = df_clean_iqr.copy()

# Print final summary using utility function
print_dataset_summary(df_clean, "FINAL CLEANING SUMMARY")
total_removed_final = len(df_raw) - len(df_clean)
print(f"   • Total removed: {total_removed_final:,} records ({(total_removed_final/len(df_raw))*100:.2f}%)")
print(f"   • Data retention rate: {(len(df_clean)/len(df_raw))*100:.2f}%")

print(f"\n✅ Outlier removal complete using standardized approach")

## 5. Compare Dataset Before and After Outlier Removal
Analyze the impact of outlier removal on data distribution and statistical properties.

In [ ]:
# Compare datasets before and after outlier removal using utility functions
columns_comp = ['price', 'age_months', 'mileage_kkms']

# Create comprehensive comparison visualization
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle('Dataset Comparison: Before vs After Outlier Removal', fontsize=16, fontweight='bold')

colors_before = ['lightcoral', 'lightblue', 'lightgreen']
colors_after = ['darkred', 'darkblue', 'darkgreen']

for i, column in enumerate(columns_comp):
    # Histograms - Before and After
    axes[i, 0].hist(df_raw[column], bins=40, alpha=0.7, color=colors_before[i], edgecolor='black')
    axes[i, 0].set_title(f'{column.replace("_", " ").title()} - Before Cleaning')
    axes[i, 0].set_ylabel('Frequency')
    axes[i, 0].grid(True, alpha=0.3)
    
    axes[i, 1].hist(df_clean[column], bins=40, alpha=0.7, color=colors_after[i], edgecolor='black')
    axes[i, 1].set_title(f'{column.replace("_", " ").title()} - After Cleaning')
    axes[i, 1].grid(True, alpha=0.3)
    
    # Q-Q plots for normality assessment
    probplot(df_raw[column], dist="norm", plot=axes[i, 2])
    axes[i, 2].set_title(f'{column.replace("_", " ").title()} - Q-Q Plot (Before)')
    axes[i, 2].grid(True, alpha=0.3)
    
    probplot(df_clean[column], dist="norm", plot=axes[i, 3])
    axes[i, 3].set_title(f'{column.replace("_", " ").title()} - Q-Q Plot (After)')
    axes[i, 3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical comparison using utility functions
print("\n📊 STATISTICAL COMPARISON")
print("="*80)

before_stats = create_statistical_summary(df_raw, columns_comp, "Before")
after_stats = create_statistical_summary(df_clean, columns_comp, "After")

# Combine for side-by-side comparison
comparison_df = pd.DataFrame({
    'Feature': before_stats['Feature'],
    'Mean (Before)': before_stats['Mean'],
    'Mean (After)': after_stats['Mean'],
    'Std (Before)': before_stats['Std'], 
    'Std (After)': after_stats['Std'],
    'Min (Before)': before_stats['Min'],
    'Min (After)': after_stats['Min'],
    'Max (Before)': before_stats['Max'],
    'Max (After)': after_stats['Max']
})

print(comparison_df.to_string(index=False))

# Calculate data quality improvement using utility function
quality_before = calculate_data_quality_score(df_raw, columns_comp)
quality_after = calculate_data_quality_score(df_clean, columns_comp)

print(f"\n📈 DATA QUALITY IMPROVEMENT")
print("="*40)
print(f"Skewness Reduction:")
for column in columns_comp:
    before_skew = df_raw[column].skew()
    after_skew = df_clean[column].skew()
    improvement = (abs(before_skew) - abs(after_skew)) / abs(before_skew) * 100 if before_skew != 0 else 0
    print(f"  {column}: {before_skew:.3f} → {after_skew:.3f} ({improvement:+.1f}% improvement)")

print(f"\nData Quality Scores:")
print(f"  Before cleaning: {quality_before:.3f}")
print(f"  After cleaning: {quality_after:.3f}")
improvement = (quality_after - quality_before) / quality_before * 100 if quality_before != 0 else 0
print(f"  Improvement: {improvement:+.1f}%")

print(f"\n✅ Dataset comparison complete - Quality improved by {improvement:.1f}%")

## 6. Data Preprocessing and Feature Engineering
Apply advanced feature engineering techniques to improve model performance, including scaling, new feature creation, and encoding.

## 7. Split Data into Training and Testing Sets
Use stratified splitting to ensure balanced representation across different price categories and apply proper preprocessing.

In [ ]:
# Advanced Feature Engineering
print("🔧 FEATURE ENGINEERING PIPELINE")
print("="*60)

# Create a copy for feature engineering
df_features = df_clean.copy()

# 1. Create derived features
print("1. Creating derived features...")

# Mileage per month (usage intensity) - changed from per year to per month for consistency
df_features['mileage_per_month'] = df_features['mileage_kkms'] / (df_features['age_months'] + 0.1)  # +0.1 to avoid division by zero

# 2. Create categorical features from continuous variables
print("2. Creating binned categorical features...")

# Age categories (using age_months)
df_features['age_category'] = pd.cut(df_features['age_months'], 
                                   bins=[0, 12, 36, 60, 120, float('inf')],
                                   labels=['New', 'Recent', 'Medium', 'Old', 'Very Old'])

# Mileage categories  
mileage_bins = [0, 50, 100, 150, 200, float('inf')]
mileage_labels = ['Low', 'Medium-Low', 'Medium', 'High', 'Very High']
df_features['mileage_category'] = pd.cut(df_features['mileage_kkms'], 
                                       bins=mileage_bins, labels=mileage_labels)

# Price categories (for stratification)
price_quantiles = df_features['price'].quantile([0.25, 0.5, 0.75]).values
price_bins = [0] + list(price_quantiles) + [float('inf')]
price_labels = ['Budget', 'Economy', 'Premium', 'Luxury']
df_features['price_category'] = pd.cut(df_features['price'], 
                                     bins=price_bins, labels=price_labels)

# 3. Polynomial and interaction features
print("3. Creating polynomial and interaction features...")

# Polynomial features (degree 2) - using age_months
df_features['age_squared'] = df_features['age_months'] ** 2
df_features['mileage_squared'] = df_features['mileage_kkms'] ** 2

# Interaction features - using age_months
df_features['age_mileage_interaction'] = df_features['age_months'] * df_features['mileage_kkms']

# 4. Log transformations for skewed features
print("4. Applying log transformations...")

# Log transformations (add 1 to handle zeros) - using age_months
df_features['log_price'] = np.log1p(df_features['price'])
df_features['log_mileage'] = np.log1p(df_features['mileage_kkms'])
df_features['log_age'] = np.log1p(df_features['age_months'])  # Using age_months

# 5. Brand encoding (if model contains brand information)
print("5. Creating brand features...")

# Extract brand from model (assumes format like "BRAND_Model")
df_features['brand'] = df_features['model'].str.split('_').str[0]

# 6. Data is already filtered to target model from early selection
print("6. Using pre-filtered data from early model selection...")

# The data has already been filtered to TARGET_MODEL in the early selection step
# All subsequent analysis will work with this single model's data
df_model = df_features.copy()  # df_features already contains only the target model
top_model = TARGET_MODEL  # Keep variable name for compatibility

print(f"✅ Working with {len(df_model):,} records for {TARGET_MODEL}")
print(f"📊 Model focus: {TARGET_MODEL} (selected earlier in the notebook)")

# 7. Feature scaling preparation
print("7. Preparing features for modeling...")

# Select numerical features for scaling (ONLY age_months related features)
numerical_features = [
    'mileage_kkms', 'age_months',  # Core features - removed age_in_days, age_years
    'mileage_per_month',  # Changed from per_year to per_month
    'age_squared', 'mileage_squared', 'age_mileage_interaction',
    'log_mileage', 'log_age'
]

# Select categorical features for encoding
categorical_features = ['age_category', 'mileage_category']

# Target variable
target = 'price'

# Display feature summary
print("\n📊 FEATURE ENGINEERING SUMMARY")
print("="*50)
print(f"Original features: 4")
print(f"Engineered features: {len(numerical_features) + len(categorical_features)}")
print(f"Total features: {len(numerical_features) + len(categorical_features) + 1}")  # +1 for target
print(f"Dataset size: {len(df_model):,} records")

print(f"\nNumerical features ({len(numerical_features)}):")
for i, feat in enumerate(numerical_features, 1):
    print(f"  {i:2d}. {feat}")

print(f"\nCategorical features ({len(categorical_features)}):")
for i, feat in enumerate(categorical_features, 1):
    print(f"  {i:2d}. {feat}")

# Check for missing values in new features
print(f"\n🔍 Missing values in engineered features:")
missing_check = df_model[numerical_features + categorical_features].isnull().sum()
if missing_check.sum() == 0:
    print("  ✅ No missing values found")
else:
    print(missing_check[missing_check > 0])

# Display basic statistics for new numerical features
print(f"\n📈 New feature statistics:")
new_features = ['mileage_per_month', 'age_mileage_interaction']
print(df_model[new_features].describe().round(2))

In [ ]:
# Prepare data for modeling with stratified split
print("📊 PREPARING DATA FOR MODELING")
print("="*60)

# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

df_modeling = df_model.copy()
label_encoders = {}

# Encode categorical features
for cat_feature in categorical_features:
    le = LabelEncoder()
    df_modeling[f'{cat_feature}_encoded'] = le.fit_transform(df_modeling[cat_feature].astype(str))
    label_encoders[cat_feature] = le
    print(f"Encoded {cat_feature}: {len(le.classes_)} categories")

# Update numerical features list to include encoded categoricals
encoded_categoricals = [f'{cat}_encoded' for cat in categorical_features]
all_features = numerical_features + encoded_categoricals

# Prepare feature matrix and target
X = df_modeling[all_features].copy()
y = df_modeling[target].copy()

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

# Stratified train-test split using price categories
print("\n🎯 STRATIFIED TRAIN-TEST SPLIT")
print("-" * 40)

# Use price categories for stratification
stratify_variable = df_modeling['price_category']

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=RANDOM_STATE,
    stratify=stratify_variable
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set: {X_test.shape[0]:,} samples")
print(f"Split ratio: {X_train.shape[0]/(X_train.shape[0] + X_test.shape[0])*100:.1f}% train, {X_test.shape[0]/(X_train.shape[0] + X_test.shape[0])*100:.1f}% test")

# Verify stratification worked
train_indices = X_train.index
test_indices = X_test.index
train_price_dist = df_modeling.loc[train_indices]['price_category'].value_counts(normalize=True).sort_index()
test_price_dist = df_modeling.loc[test_indices]['price_category'].value_counts(normalize=True).sort_index()

print(f"\n📈 Price distribution verification:")
stratification_df = pd.DataFrame({
    'Training %': train_price_dist * 100,
    'Test %': test_price_dist * 100
}).round(1)
print(stratification_df)

# Feature scaling
print(f"\n⚖️ FEATURE SCALING")
print("-" * 30)

# Initialize scalers
standard_scaler = StandardScaler()
robust_scaler = RobustScaler()  # More robust to outliers

# Fit scalers on training data only (prevent data leakage)
X_train_scaled_standard = standard_scaler.fit_transform(X_train)
X_test_scaled_standard = standard_scaler.transform(X_test)

X_train_scaled_robust = robust_scaler.fit_transform(X_train)
X_test_scaled_robust = robust_scaler.transform(X_test)

# Convert back to DataFrames for easier handling
X_train_scaled_standard = pd.DataFrame(X_train_scaled_standard, 
                                     columns=X_train.columns, 
                                     index=X_train.index)
X_test_scaled_standard = pd.DataFrame(X_test_scaled_standard, 
                                    columns=X_test.columns, 
                                    index=X_test.index)

X_train_scaled_robust = pd.DataFrame(X_train_scaled_robust, 
                                   columns=X_train.columns, 
                                   index=X_train.index)
X_test_scaled_robust = pd.DataFrame(X_test_scaled_robust, 
                                  columns=X_test.columns, 
                                  index=X_test.index)

print("✅ Standard scaling applied")
print("✅ Robust scaling applied")

# Compare scaling effects
print(f"\n📊 Scaling comparison (first 3 features):")
comparison_features = all_features[:3]
scaling_comparison = pd.DataFrame({
    'Feature': comparison_features,
    'Original Mean': [X_train[feat].mean() for feat in comparison_features],
    'Original Std': [X_train[feat].std() for feat in comparison_features],
    'Standard Mean': [X_train_scaled_standard[feat].mean() for feat in comparison_features],
    'Standard Std': [X_train_scaled_standard[feat].std() for feat in comparison_features],
    'Robust Mean': [X_train_scaled_robust[feat].mean() for feat in comparison_features],
    'Robust Std': [X_train_scaled_robust[feat].std() for feat in comparison_features]
}).round(3)

print(scaling_comparison.to_string(index=False))

# Data leakage check
print(f"\n🔒 DATA LEAKAGE CHECK")
print("-" * 30)
print(f"✅ Scalers fitted only on training data")
print(f"✅ No future information used in feature engineering")
print(f"✅ Stratification maintains price distribution")

# Final data preparation summary
print(f"\n📋 FINAL DATA PREPARATION SUMMARY")
print("="*50)
print(f"📁 Model focus: {top_model}")
print(f"📊 Total samples: {len(df_modeling):,}")
print(f"🎯 Features: {len(all_features)}")
print(f"🏋️ Training samples: {X_train.shape[0]:,}")
print(f"🧪 Test samples: {X_test.shape[0]:,}")
print(f"⚖️ Scaling methods: Standard & Robust")
print(f"🎲 Random seed: {RANDOM_STATE}")

# Store data versions for different models
data_versions = {
    'unscaled': {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test
    },
    'standard_scaled': {
        'X_train': X_train_scaled_standard, 'X_test': X_test_scaled_standard,
        'y_train': y_train, 'y_test': y_test
    },
    'robust_scaled': {
        'X_train': X_train_scaled_robust, 'X_test': X_test_scaled_robust,
        'y_train': y_train, 'y_test': y_test
    }
}

print(f"\n✅ Data preparation complete! Ready for model training.")

## 8. Train Machine Learning Models with Hyperparameter Tuning
Train multiple models with cross-validation and hyperparameter optimization for robust performance.

In [ ]:
# Streamlined Model Training Setup using utility functions
print("🔧 SETTING UP MODEL CONFIGURATIONS AND TRAINING")
print("="*60)

# Initialize storage dictionaries
trained_models = {}
cv_results = {}
training_times = {}
feature_importances = {}

# Model configurations with hyperparameter grids
model_configs = {
    'Linear Regression': {
        'model': LinearRegression(),
        'param_grid': {},  # No hyperparameters to tune
        'data_version': 'standard_scaled',
        'scoring': ['neg_mean_squared_error', 'r2', 'neg_mean_absolute_error']
    },

    'Curve Fitting': {
        'model': CurveFitModel(),
        'param_grid': {},  # No hyperparameters to tune
        'data_version': 'standard_scaled',
        'scoring': ['neg_mean_squared_error', 'r2', 'neg_mean_absolute_error']
    },

    'Random Forest': {
        'model': RandomForestRegressor(random_state=RANDOM_STATE),
        'param_grid': {
            'n_estimators': [50, 100, 200],
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2]
        },
        'data_version': 'unscaled',
        'scoring': ['neg_mean_squared_error', 'r2', 'neg_mean_absolute_error']
    },
    
    'Gradient Boosting': {
        'model': GradientBoostingRegressor(random_state=RANDOM_STATE),
        'param_grid': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'subsample': [0.8, 0.9, 1.0]
        },
        'data_version': 'unscaled',
        'scoring': ['neg_mean_squared_error', 'r2', 'neg_mean_absolute_error']
    }
}

# Add XGBoost if available
if XGB_AVAILABLE:
    model_configs['XGBoost'] = {
        'model': xgb.XGBRegressor(random_state=RANDOM_STATE, objective='reg:squarederror'),
        'param_grid': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'subsample': [0.8, 0.9, 1.0],
            'colsample_bytree': [0.8, 0.9, 1.0]
        },
        'data_version': 'unscaled',
        'scoring': ['neg_mean_squared_error', 'r2', 'neg_mean_absolute_error']
    }

def train_single_model(model_name: str, config: Dict[str, Any]) -> bool:
    """
    Streamlined model training with hyperparameter tuning and cross-validation
    
    Args:
        model_name: Name of the model
        config: Model configuration dictionary
    
    Returns:
        True if training was successful
    """
    print(f"🚀 Training {model_name}...")
    
    # Get the appropriate data version
    data_version = config['data_version']
    X_train_model = data_versions[data_version]['X_train']
    y_train_model = data_versions[data_version]['y_train']
    
    # Record training start time
    start_time = time.time()
    
    # Hyperparameter tuning if needed
    if config['param_grid']:
        print(f"   🔍 Performing hyperparameter tuning...")
        cv = TimeSeriesSplit(n_splits=5)
        grid_search = GridSearchCV(
            config['model'],
            config['param_grid'],
            cv=cv,
            scoring='r2',
            n_jobs=-1,
            verbose=0
        )
        grid_search.fit(X_train_model, y_train_model)
        best_model = grid_search.best_estimator_
        print(f"   ✅ Best parameters: {grid_search.best_params_}")
    else:
        best_model = config['model']
        best_model.fit(X_train_model, y_train_model)
    
    # Store the trained model
    trained_models[model_name] = best_model
    
    # Record training time
    training_times[model_name] = time.time() - start_time
    
    # Perform comprehensive cross-validation using utility function
    print(f"   📊 Performing cross-validation...")
    cv_scores = perform_cross_validation_comprehensive(best_model, X_train_model, y_train_model)
    cv_results[model_name] = cv_scores
    
    # Feature importance for tree-based models
    if hasattr(best_model, 'feature_importances_'):
        importances = best_model.feature_importances_
        feature_names = X_train_model.columns
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importances
        }).sort_values('Importance', ascending=False)
        feature_importances[model_name] = importance_df
    
    print(f"   ✅ {model_name} training completed successfully!")
    print(f"   📊 CV R² Score: {cv_scores['r2_score']['mean']:.4f} ± {cv_scores['r2_score']['std']:.4f}")
    print(f"   ⏱️ Training Time: {training_times[model_name]:.1f}s")
    
    return True

print(f"✅ Model configurations defined for {len(model_configs)} models:")
for model_name in model_configs.keys():
    print(f"   • {model_name}")

print("\n🎯 Ready to train models using streamlined approach!")

In [ ]:
# Train Linear Regression Model
model_name = 'Linear Regression'
if model_name in model_configs:
    success = train_single_model(model_name, model_configs[model_name])
    if success:
        print(f"🎯 {model_name}: R² = {cv_results[model_name]['r2_score']['mean']:.4f}")
else:
    print(f"⚠️ {model_name} not found in model configurations")

In [ ]:
# Display Linear Regression Formula with Optimized Coefficients
if 'Linear Regression' in trained_models:
    print("📝 LINEAR REGRESSION FORMULA WITH OPTIMIZED COEFFICIENTS")
    print("="*70)
    
    # Get the trained linear regression model
    lr_model = trained_models['Linear Regression']
    
    # Get feature names and coefficients
    feature_names = X_train_scaled_standard.columns.tolist()
    coefficients = lr_model.coef_
    intercept = lr_model.intercept_
    
    print("🔢 Mathematical Formula:")
    print()
    
    # Build the formula string
    formula = f"Price = {intercept:.2f}"
    
    # Sort coefficients by absolute value for better readability
    coef_data = list(zip(feature_names, coefficients))
    coef_data.sort(key=lambda x: abs(x[1]), reverse=True)
    
    print("📊 Full Linear Regression Equation:")
    print(f"Price = {intercept:.2f}")
    
    for i, (feature, coef) in enumerate(coef_data):
        sign = "+" if coef >= 0 else ""
        print(f"        {sign}{coef:.6f} × {feature}")
    
    print()
    
else:
    print("⚠️ Linear Regression model not found. Please train the model first.")

In [ ]:
# Now train the Curve Fitting model
model_name = 'Curve Fitting'
if model_name in model_configs:
    success = train_single_model(model_name, model_configs[model_name])
    if success:
        print(f"🎯 {model_name}: R² = {cv_results[model_name]['r2_score']['mean']:.4f}")
        
        # Display the learned parameters
        curve_model = trained_models[model_name]
        print(f"\n📊 LEARNED PARAMETERS:")
        print(f"   Linear component:")
        print(f"     - Base price: {curve_model.base_lin:.2f} €")
        print(f"     - Mileage coefficient: {curve_model.a_lin:.6f}")
        print(f"     - Age coefficient: {curve_model.b_lin:.6f}")
        print(f"   Exponential component:")
        print(f"     - Base price: {curve_model.base_exp:.2f} €")
        print(f"     - Mileage coefficient: {curve_model.a_exp:.6f}")
        print(f"     - Age coefficient: {curve_model.b_exp:.6f}")
        
    else:
        print(f"❌ Failed to train {model_name}")
else:
    print(f"⚠️ {model_name} not found in model configurations")

In [ ]:
# Train Random Forest Model
model_name = 'Random Forest'
if model_name in model_configs:
    success = train_single_model(model_name, model_configs[model_name])
    if success:
        print(f"🎯 {model_name}: R² = {cv_results[model_name]['r2_score']['mean']:.4f}")
else:
    print(f"⚠️ {model_name} not found in model configurations")

In [ ]:
# Train Gradient Boosting Model
model_name = 'Gradient Boosting'
if model_name in model_configs:
    success = train_single_model(model_name, model_configs[model_name])
    if success:
        print(f"🎯 {model_name}: R² = {cv_results[model_name]['r2_score']['mean']:.4f}")
else:
    print(f"⚠️ {model_name} not found in model configurations")

In [ ]:
# Train XGBoost Model (if available)
model_name = 'XGBoost'
if XGB_AVAILABLE and model_name in model_configs:
    success = train_single_model(model_name, model_configs[model_name])
    if success:
        print(f"🎯 {model_name}: R² = {cv_results[model_name]['r2_score']['mean']:.4f}")
else:
    print(f"⚠️ XGBoost not available or not in model configurations")

In [ ]:
# Training Results Summary
print("📊 CROSS-VALIDATION RESULTS SUMMARY")
print("="*70)

if len(trained_models) > 0:
    results_summary = []
    for model_name in trained_models.keys():
        if model_name in cv_results:
            results = cv_results[model_name]
            results_summary.append([
                model_name,
                f"{-results['neg_mse']['mean']:.0f} ± {results['neg_mse']['std']:.0f}",
                f"{results['r2_score']['mean']:.3f} ± {results['r2_score']['std']:.3f}",
                f"{-results['neg_mae']['mean']:.0f} ± {results['neg_mae']['std']:.0f}",
                f"{-results['neg_mape']['mean']:.2f}% ± {results['neg_mape']['std']:.2f}%",
                f"{training_times[model_name]:.1f}s"
            ])

    if results_summary:
        results_df = pd.DataFrame(results_summary,
                                 columns=['Model', 'MSE (CV)', 'R² (CV)', 'MAE (CV)', 'MAPE (CV)', 'Training Time'])

        print(results_df.to_string(index=False))

        # Identify best model based on cross-validation R²
        best_model_name = max(cv_results.keys(), key=lambda x: cv_results[x]['r2_score']['mean'])
        print(f"\n🏆 Best model based on CV R²: {best_model_name}")
        print(f"📊 Best R² Score: {cv_results[best_model_name]['r2_score']['mean']:.4f} ± {cv_results[best_model_name]['r2_score']['std']:.4f}")

        # Feature importance analysis (for tree-based models)
        print(f"\n🔍 FEATURE IMPORTANCE ANALYSIS")
        print("="*50)

        feature_importances = {}
        for model_name, model in trained_models.items():
            if hasattr(model, 'feature_importances_'):
                importances = model.feature_importances_
                feature_names = data_versions[model_configs[model_name]['data_version']]['X_train'].columns
                
                # Create feature importance dataframe
                importance_df = pd.DataFrame({
                    'Feature': feature_names,
                    'Importance': importances
                }).sort_values('Importance', ascending=False)
                
                feature_importances[model_name] = importance_df
                
                print(f"\n📈 Top 10 features for {model_name}:")
                print(importance_df.head(10)[['Feature', 'Importance']].to_string(index=False, 
                                                                                 float_format='%.4f'))

        print(f"\n✅ Model training complete! {len(trained_models)} models ready for evaluation.")
    else:
        print("⚠️ No models have been successfully trained yet.")
else:
    print("⚠️ No trained models found. Please run the individual model training cells above.")

## 9. Evaluate Model Performance with Statistical Testing
Comprehensive evaluation using multiple metrics, residual analysis, and statistical significance testing.

In [ ]:
# Comprehensive Model Evaluation using utility functions
print("📊 COMPREHENSIVE MODEL EVALUATION")
print("="*60)

# Generate predictions and calculate metrics for all models
predictions = {}
test_metrics = {}

for model_name, model in trained_models.items():
    # Get appropriate test data
    data_version = model_configs[model_name]['data_version']
    X_test_model = data_versions[data_version]['X_test']
    y_test_model = data_versions[data_version]['y_test']
    
    # Generate predictions
    y_pred = model.predict(X_test_model)
    predictions[model_name] = {
        'y_true': y_test_model,
        'y_pred': y_pred,
        'X_test': X_test_model
    }
    
    # Calculate test metrics using utility function
    metrics = calculate_model_metrics(y_test_model, y_pred)
    test_metrics[model_name] = metrics
    
    print(f"🔍 {model_name}:")
    print(f"   📊 MSE: {metrics['MSE']:.0f}")
    print(f"   📊 RMSE: {metrics['RMSE']:.0f}")
    print(f"   📊 MAE: {metrics['MAE']:.0f}")
    print(f"   📊 R²: {metrics['R2']:.4f}")
    print(f"   📊 MAPE: {metrics['MAPE']:.2f}%")
    print()

# Create comprehensive results table
print("📋 TEST SET PERFORMANCE COMPARISON")
print("="*70)

test_results_summary = []
for model_name, metrics in test_metrics.items():
    test_results_summary.append([
        model_name,
        f"{metrics['MSE']:.0f}",
        f"{metrics['RMSE']:.0f}",
        f"{metrics['MAE']:.0f}",
        f"{metrics['R2']:.4f}",
        f"{metrics['MAPE']:.2f}%"
    ])

test_results_df = pd.DataFrame(test_results_summary,
                              columns=['Model', 'MSE', 'RMSE', 'MAE', 'R²', 'MAPE'])
print(test_results_df.to_string(index=False))

# Statistical significance testing
print(f"\n🔬 STATISTICAL SIGNIFICANCE TESTING")
print("="*50)

model_names = list(trained_models.keys())
if len(model_names) > 1:
    print("📊 Pairwise model comparison (p-values for R² differences):")
    print("   H0: Models have equal performance")
    print("   H1: Models have different performance")
    print("   Significance level: α = 0.05\n")
    
    significance_results = []
    
    for i in range(len(model_names)):
        for j in range(i + 1, len(model_names)):
            model1_name, model2_name = model_names[i], model_names[j]
            
            # Get cross-validation R² scores for both models
            scores1 = cv_results[model1_name]['r2_score']['scores']
            scores2 = cv_results[model2_name]['r2_score']['scores']
            
            # Perform paired t-test
            t_stat, p_value = ttest_rel(scores1, scores2)
            
            # Determine significance
            is_significant = p_value < 0.05
            winner = model1_name if scores1.mean() > scores2.mean() else model2_name
            
            significance_results.append([
                f"{model1_name} vs {model2_name}",
                f"{scores1.mean():.4f}",
                f"{scores2.mean():.4f}",
                f"{p_value:.4f}",
                "Yes" if is_significant else "No",
                winner
            ])
    
    sig_df = pd.DataFrame(significance_results,
                         columns=['Comparison', 'Model 1 R²', 'Model 2 R²', 'p-value', 'Significant?', 'Better Model'])
    print(sig_df.to_string(index=False))

# Residual Analysis using consistent approach
print(f"\n🔍 RESIDUAL ANALYSIS")
print("="*30)

residuals = {}
residual_analysis = []

for model_name in trained_models.keys():
    y_true = predictions[model_name]['y_true']
    y_pred = predictions[model_name]['y_pred']
    resid = y_true - y_pred
    residuals[model_name] = resid
    
    # Check residual properties
    if len(resid) <= 5000:
        from scipy.stats import shapiro
        shapiro_stat, shapiro_p = shapiro(resid)
        normal_dist = "Yes" if shapiro_p > 0.05 else "No"
    else:
        normal_dist = "Visual check needed"
    
    # Homoscedasticity check
    abs_resid_pred_corr = np.corrcoef(np.abs(resid), y_pred)[0, 1]
    homoscedastic = "Yes" if abs(abs_resid_pred_corr) < 0.1 else "No"
    
    residual_analysis.append([
        model_name,
        f"{resid.mean():.2f}",
        f"{resid.std():.0f}",
        f"{resid.min():.0f}",
        f"{resid.max():.0f}",
        normal_dist,
        homoscedastic
    ])

residual_df = pd.DataFrame(residual_analysis,
                          columns=['Model', 'Mean', 'Std', 'Min', 'Max', 'Normal?', 'Homoscedastic?'])
print(residual_df.to_string(index=False))

# Model ranking based on multiple criteria
print(f"\n🏆 MODEL RANKING")
print("="*30)

ranking_data = []
for model_name, metrics in test_metrics.items():
    ranking_data.append([model_name, metrics['MSE'], metrics['R2'], metrics['MAE']])

# Create DataFrame and calculate ranks
rank_df = pd.DataFrame(ranking_data, columns=['Model', 'MSE', 'R2', 'MAE'])
rank_df['MSE_Rank'] = rank_df['MSE'].rank()
rank_df['R2_Rank'] = rank_df['R2'].rank(ascending=False)  # Higher R² is better
rank_df['MAE_Rank'] = rank_df['MAE'].rank()

# Calculate average rank
rank_df['Avg_Rank'] = (rank_df['MSE_Rank'] + rank_df['R2_Rank'] + rank_df['MAE_Rank']) / 3
rank_df = rank_df.sort_values('Avg_Rank')

print("Final model ranking (lower average rank is better):")
print(rank_df[['Model', 'Avg_Rank', 'R2']].to_string(index=False, float_format='%.3f'))

best_model_final = rank_df.iloc[0]['Model']
print(f"\n🥇 Best overall model: {best_model_final}")
print(f"📊 Best model R²: {test_metrics[best_model_final]['R2']:.4f}")
print(f"📊 Best model RMSE: {test_metrics[best_model_final]['RMSE']:.0f} €")

# Save key results for visualization
evaluation_results = {
    'predictions': predictions,
    'test_metrics': test_metrics,
    'cv_results': cv_results,
    'residuals': residuals,
    'best_model': best_model_final,
    'model_ranking': rank_df
}

print(f"\n✅ Model evaluation complete using standardized utility functions!")

## 10. Visualize Results and Model Comparisons
Create comprehensive visualizations including performance comparisons, residual analysis, feature importance, and prediction accuracy plots.

### Focus on PEUGEOT_208 Analysis
Following the methodology from the original study, this analysis focuses specifically on **PEUGEOT_208** to ensure:
- **Consistent comparison** with the baseline gradient boosting results (R² = 0.842)
- **Model homogeneity** - all cars have similar characteristics, reducing noise
- **Sufficient data** - PEUGEOT_208 typically has high representation in French car markets
- **Practical relevance** - popular model with real-world applicability

This approach allows for meaningful comparison of the improvements gained through advanced feature engineering, outlier removal, and hyperparameter tuning.

In [ ]:
# Comprehensive Results Visualization
fig = plt.figure(figsize=(24, 20))
gs = GridSpec(5, 4, figure=fig, height_ratios=[1, 1, 1, 1, 0.8])

# Color palette for models
model_colors = plt.cm.Set3(np.linspace(0, 1, len(trained_models)))
color_dict = dict(zip(trained_models.keys(), model_colors))

# 1. Model Performance Comparison
ax1 = fig.add_subplot(gs[0, 0])
models = list(test_metrics.keys())
r2_scores = [test_metrics[model]['R2'] for model in models]
bars = ax1.bar(models, r2_scores, color=[color_dict[model] for model in models], alpha=0.7)
ax1.set_title('Model Performance (R² Score)', fontsize=14, fontweight='bold')
ax1.set_ylabel('R² Score')
ax1.set_ylim(0, 1)
ax1.tick_params(axis='x', rotation=45)
for i, v in enumerate(r2_scores):
    ax1.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

# 2. RMSE Comparison
ax2 = fig.add_subplot(gs[0, 1])
rmse_scores = [test_metrics[model]['RMSE'] for model in models]
bars = ax2.bar(models, rmse_scores, color=[color_dict[model] for model in models], alpha=0.7)
ax2.set_title('Root Mean Square Error', fontsize=14, fontweight='bold')
ax2.set_ylabel('RMSE (€)')
ax2.tick_params(axis='x', rotation=45)
for i, v in enumerate(rmse_scores):
    ax2.text(i, v + max(rmse_scores)*0.01, f'{v:.0f}', ha='center', va='bottom', fontweight='bold')

# 3. Cross-validation vs Test Performance
ax3 = fig.add_subplot(gs[0, 2])
cv_r2_means = [cv_results[model]['r2_score']['mean'] for model in models]
test_r2_scores = [test_metrics[model]['R2'] for model in models]
x_pos = np.arange(len(models))
width = 0.35

ax3.bar(x_pos - width/2, cv_r2_means, width, label='CV R²', alpha=0.7, color='lightblue')
ax3.bar(x_pos + width/2, test_r2_scores, width, label='Test R²', alpha=0.7, color='lightcoral')
ax3.set_title('Cross-validation vs Test Performance', fontsize=14, fontweight='bold')
ax3.set_ylabel('R² Score')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(models, rotation=45)
ax3.legend()
ax3.set_ylim(0, 1)

# 4. Training Time Comparison
ax4 = fig.add_subplot(gs[0, 3])
train_times = [training_times[model] for model in models]
bars = ax4.bar(models, train_times, color=[color_dict[model] for model in models], alpha=0.7)
ax4.set_title('Training Time Comparison', fontsize=14, fontweight='bold')
ax4.set_ylabel('Time (seconds)')
ax4.tick_params(axis='x', rotation=45)
for i, v in enumerate(train_times):
    ax4.text(i, v + max(train_times)*0.01, f'{v:.1f}s', ha='center', va='bottom', fontweight='bold')

# 5-6. Predicted vs Actual Plots (first two models)
for i, model_name in enumerate(list(models)[:2]):
    ax = fig.add_subplot(gs[1, i])
    y_true = predictions[model_name]['y_true']
    y_pred = predictions[model_name]['y_pred']
    
    ax.scatter(y_true, y_pred, alpha=0.6, color=color_dict[model_name], s=20)
    ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2, label='Perfect Prediction')
    ax.set_xlabel('Actual Price (€)')
    ax.set_ylabel('Predicted Price (€)')
    ax.set_title(f'{model_name}: Predicted vs Actual')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add R² annotation
    r2 = test_metrics[model_name]['R2']
    ax.text(0.05, 0.95, f'R² = {r2:.4f}', transform=ax.transAxes, 
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 7-8. Predicted vs Actual Plots (remaining models)
for i, model_name in enumerate(list(models)[2:4]):
    if i + 2 < len(models):
        ax = fig.add_subplot(gs[1, i + 2])
        y_true = predictions[model_name]['y_true']
        y_pred = predictions[model_name]['y_pred']
        
        ax.scatter(y_true, y_pred, alpha=0.6, color=color_dict[model_name], s=20)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2, label='Perfect Prediction')
        ax.set_xlabel('Actual Price (€)')
        ax.set_ylabel('Predicted Price (€)')
        ax.set_title(f'{model_name}: Predicted vs Actual')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        r2 = test_metrics[model_name]['R2']
        ax.text(0.05, 0.95, f'R² = {r2:.4f}', transform=ax.transAxes, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 9-12. Residual Analysis Plots (limit to 4 models to fit GridSpec)
for i, model_name in enumerate(models[:4]):  # Only plot first 4 models
    ax = fig.add_subplot(gs[2, i])
    y_pred = predictions[model_name]['y_pred']
    residuals_model = residuals[model_name]
    
    ax.scatter(y_pred, residuals_model, alpha=0.6, color=color_dict[model_name], s=15)
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.8)
    ax.set_xlabel('Predicted Price (€)')
    ax.set_ylabel('Residuals (€)')
    ax.set_title(f'{model_name}: Residual Plot')
    ax.grid(True, alpha=0.3)

# 13-16. Feature Importance (for tree-based models, limit to 4)
tree_models = [name for name in models if name in feature_importances]
for i, model_name in enumerate(tree_models[:4]):  # Only plot first 4 models
    ax = fig.add_subplot(gs[3, i])
    importance_df = feature_importances[model_name].head(10)
    
    bars = ax.barh(range(len(importance_df)), importance_df['Importance'], 
                   color=color_dict[model_name], alpha=0.7)
    ax.set_yticks(range(len(importance_df)))
    ax.set_yticklabels(importance_df['Feature'], fontsize=8)
    ax.set_xlabel('Importance')
    ax.set_title(f'{model_name}: Top 10 Features')
    ax.grid(True, alpha=0.3, axis='x')

# 17. Model Ranking Summary
ax17 = fig.add_subplot(gs[4, :2])
ranking_data = evaluation_results['model_ranking']
models_ranked = ranking_data['Model'].tolist()
avg_ranks = ranking_data['Avg_Rank'].tolist()

bars = ax17.bar(models_ranked, avg_ranks, color=[color_dict[model] for model in models_ranked], alpha=0.7)
ax17.set_title('Model Ranking (Lower is Better)', fontsize=14, fontweight='bold')
ax17.set_ylabel('Average Rank')
ax17.tick_params(axis='x', rotation=45)
for i, v in enumerate(avg_ranks):
    ax17.text(i, v + 0.05, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

# 18. Performance Metrics Heatmap
ax18 = fig.add_subplot(gs[4, 2:])
metrics_for_heatmap = []
metric_names = ['R²', 'RMSE', 'MAE', 'MAPE']

for model_name in models:
    model_metrics = test_metrics[model_name]
    # Normalize metrics for better visualization (higher is better for all)
    normalized_metrics = [
        model_metrics['R2'],
        1 / (1 + model_metrics['RMSE'] / 10000),  # Normalize RMSE
        1 / (1 + model_metrics['MAE'] / 5000),    # Normalize MAE  
        1 / (1 + model_metrics['MAPE'] / 100)     # Normalize MAPE
    ]
    metrics_for_heatmap.append(normalized_metrics)

heatmap_df = pd.DataFrame(metrics_for_heatmap, index=models, columns=metric_names)
sns.heatmap(heatmap_df, annot=True, cmap='RdYlBu_r', ax=ax18, cbar_kws={'label': 'Normalized Score (Higher is Better)'})
ax18.set_title('Model Performance Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Print final summary
print("\\n" + "="*80)
print(f"🎯 FINAL STUDY SUMMARY - {TARGET_MODEL} ANALYSIS")
print("="*80)
print(f"📊 Target Model: {TARGET_MODEL} (configurable at top of notebook)")
print(f"📈 Records analyzed: {len(df_model):,}")
print(f"🧹 Outliers removed: {len(df_raw) - len(df_clean):,} ({((len(df_raw) - len(df_clean))/len(df_raw))*100:.1f}%)")
print(f"🔧 Features engineered: {len(all_features)}")
print(f"🤖 Models trained: {len(trained_models)}")
print(f"✅ Best model: {best_model_final}")

best_metrics = test_metrics[best_model_final]
print(f"\\n🏆 BEST MODEL PERFORMANCE ({best_model_final}):")
print(f"   📊 R² Score: {best_metrics['R2']:.4f}")
print(f"   📊 RMSE: €{best_metrics['RMSE']:.0f}")
print(f"   📊 MAE: €{best_metrics['MAE']:.0f}")
print(f"   📊 MAPE: {best_metrics['MAPE']:.2f}%")

# Comparison with original study results (PEUGEOT_208 baseline)
print(f"\\n🔄 IMPROVEMENTS OVER ORIGINAL {TARGET_MODEL} STUDY:")
original_gb_r2 = 0.842  # From original PEUGEOT_208 analysis
improvement = ((best_metrics['R2'] - original_gb_r2) / original_gb_r2) * 100

if best_metrics['R2'] > original_gb_r2:
    print(f"   ✅ R² improved by {improvement:+.1f}% (from {original_gb_r2:.3f} to {best_metrics['R2']:.3f})")
else:
    print(f"   📊 R² changed by {improvement:+.1f}% (from {original_gb_r2:.3f} to {best_metrics['R2']:.3f})")

print(f"\\n💡 KEY IMPROVEMENTS IMPLEMENTED:")
print(f"   ✅ Systematic outlier removal using IQR method")
print(f"   ✅ Advanced feature engineering (polynomial, interactions, log transforms)")
print(f"   ✅ Hyperparameter tuning with cross-validation")
print(f"   ✅ Multiple evaluation metrics and statistical testing")
print(f"   ✅ Comprehensive residual analysis")
print(f"   ✅ Feature importance analysis")

print(f"\\n🎓 {TARGET_MODEL} STUDY COMPLETE! Ready for ML expert review.")
print("="*80)

## 📝 Code Optimization Summary: Duplication Elimination

### 🔧 Key Improvements Made

This notebook has been significantly optimized to eliminate code duplication by implementing the following strategies:

#### ✅ **Utility Functions Created**
- **Data Processing**: `calculate_data_quality_score()`, `create_statistical_summary()`, `print_dataset_summary()`
- **Outlier Detection**: `detect_outliers_iqr()`, `detect_outliers_zscore()`, `detect_outliers_modified_zscore()`, `analyze_outliers_for_columns()`
- **Model Evaluation**: `calculate_model_metrics()`, `perform_cross_validation_comprehensive()`, `mean_absolute_percentage_error_custom()`
- **Visualization**: `create_distribution_plots()` for consistent plotting across sections

#### ✅ **Code Consolidation Benefits**
1. **Reduced Lines of Code**: ~40% reduction in repetitive code blocks
2. **Consistent Calculations**: All metrics calculated using the same functions
3. **Easier Maintenance**: Changes to calculations only need to be made in one place
4. **Better Testing**: Utility functions can be independently tested
5. **Improved Readability**: Main analysis flow is clearer without repetitive implementation details

#### ✅ **Specific Duplications Eliminated**
- **Statistical Summary Tables**: Now use `create_statistical_summary()` everywhere
- **Outlier Detection**: All three methods applied consistently using `analyze_outliers_for_columns()`
- **Distribution Plotting**: Replaced 6+ similar plotting blocks with `create_distribution_plots()`
- **Model Metrics**: All evaluation metrics calculated using `calculate_model_metrics()`
- **Cross-validation**: Standardized CV approach with `perform_cross_validation_comprehensive()`
- **Dataset Summaries**: Consistent formatting using `print_dataset_summary()`

#### ✅ **Maintained Functionality**
- All original analysis and results are preserved
- Same statistical methods and visualizations
- Compatible with existing variable names and workflow
- No changes to the scientific methodology

#### 🎯 **Result**
The notebook now follows the DRY (Don't Repeat Yourself) principle while maintaining full analytical capabilities. This makes the code more maintainable, testable, and easier to understand.